[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Delta_Lake_Hello_World.ipynb)

# Delta Lake Hello World

**Write a Delta table, update a record with MERGE, time-travel to the previous version, and clean up with VACUUM.**

Playbook: [Lakehouse Formats](../../playbooks/data/lakehouse-formats/README.md)

---

## What You Will Do

1. Install Delta Lake + PySpark (one cell)
2. Write call center data as a Delta table
3. Read it back and verify
4. MERGE — update a call status (in-progress → resolved)
5. Time travel — see the data before and after the update
6. Schema evolution — handle a new column from the source
7. VACUUM — clean up old files
8. DESCRIBE HISTORY — audit trail

**Time:** ~15 minutes

**Prerequisites:** None. Runs in Colab with no GCP project needed.

---

## 1. Setup

Install PySpark and Delta Lake. This takes ~60 seconds in Colab.

In [ ]:
# WHY: delta-spark is the Python package that adds Delta Lake support to PySpark.
# It installs both PySpark and the Delta Lake JARs.
!pip install -q delta-spark pyspark

In [ ]:
# WHY: Delta Lake needs specific Spark configurations.
# configure_spark_with_delta_pip handles the JAR (Java Archive) dependencies
# so you don't have to download and configure them manually.

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("delta-hello-world")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("Spark + Delta Lake ready")
print(f"Spark version: {spark.version}")

**You Should See:** `Spark + Delta Lake ready` with a version number. No errors.

---

## 2. Create Sample Data

Five call records from a call center. Note that call C-001 is still `in-progress` — we will update it later.

In [ ]:
from pyspark.sql import Row
from datetime import datetime

# WHY: We use simple data so you can see exactly what changes at each step.
# In production, this data comes from GCS Bronze (raw JSON/CSV files).
calls_data = [
    Row(call_id="C-001", customer_id="CUST-100", status="in-progress",
        duration=0, created_at=datetime(2026, 4, 13, 9, 0)),
    Row(call_id="C-002", customer_id="CUST-101", status="resolved",
        duration=340, created_at=datetime(2026, 4, 13, 9, 15)),
    Row(call_id="C-003", customer_id="CUST-102", status="missed",
        duration=0, created_at=datetime(2026, 4, 13, 9, 22)),
    Row(call_id="C-004", customer_id="CUST-103", status="resolved",
        duration=180, created_at=datetime(2026, 4, 13, 9, 30)),
    Row(call_id="C-005", customer_id="CUST-104", status="voicemail",
        duration=60, created_at=datetime(2026, 4, 13, 9, 45)),
]

calls_df = spark.createDataFrame(calls_data)
calls_df.show()

**You Should See:** A table with 5 rows. C-001 has `status=in-progress` and `duration=0`.

---

## 3. Write as Delta Table

This is the key difference from regular Parquet. Writing as Delta creates:
- Parquet data files (same as before)
- A `_delta_log/` directory (the transaction log — this is what makes it a Delta table)

In [ ]:
DELTA_PATH = "/tmp/delta-calls"

# WHY: format("delta") instead of format("parquet").
# That one word change gives you ACID transactions, time travel,
# MERGE, schema evolution — everything a plain Parquet folder lacks.
calls_df.write.format("delta").mode("overwrite").save(DELTA_PATH)

print(f"Delta table written to {DELTA_PATH}")

In [ ]:
# WHY: Let's see what Delta actually created on disk.
# You should see Parquet files AND a _delta_log/ directory.
import os

print("Files in Delta table directory:")
for item in sorted(os.listdir(DELTA_PATH)):
    print(f"  {item}")

print(f"\nFiles in _delta_log/:")
for item in sorted(os.listdir(os.path.join(DELTA_PATH, "_delta_log"))):
    print(f"  {item}")

**You Should See:**
- `_delta_log/` directory
- One or more `.parquet` files
- Inside `_delta_log/`: `00000000000000000000.json` — this is commit 0 (the initial write)

This is **Version 0** of the table.

---

## 4. Read the Delta Table

In [ ]:
# WHY: Reading a Delta table looks identical to reading Parquet.
# The only difference: format("delta") instead of format("parquet").
df = spark.read.format("delta").load(DELTA_PATH)
df.show()
print(f"Row count: {df.count()}")

**You Should See:** Same 5 rows you wrote. Row count: 5.

---

## 5. MERGE — Update a Call Status

Call C-001 just got resolved. The agent spent 480 seconds (8 minutes) on it.

In a regular Parquet file, you would have to:
1. Read all 5 records
2. Find C-001 and change it
3. Rewrite the entire file

With Delta Lake MERGE, you just say: "If the call_id matches, update it. If it's new, insert it."

In [ ]:
from delta.tables import DeltaTable

# The incoming update: C-001 is now resolved
update_data = [
    Row(call_id="C-001", customer_id="CUST-100", status="resolved",
        duration=480, created_at=datetime(2026, 4, 13, 9, 0)),
]
update_df = spark.createDataFrame(update_data)

# Load the Delta table for merging
delta_table = DeltaTable.forPath(spark, DELTA_PATH)

# MERGE: update if call_id matches, insert if new
(
    delta_table.alias("target")
    .merge(
        update_df.alias("source"),
        "target.call_id = source.call_id"  # match on call_id
    )
    .whenMatchedUpdateAll()   # call_id exists → update all columns
    .whenNotMatchedInsertAll() # call_id is new → insert the row
    .execute()
)

print("MERGE complete!")

In [ ]:
# Verify: C-001 should now be 'resolved' with duration=480
spark.read.format("delta").load(DELTA_PATH).show()

**You Should See:** C-001 now has `status=resolved` and `duration=480`. All other rows unchanged.

This is **Version 1** of the table.

---

## 6. Time Travel

This is the feature that makes Delta Lake special. You can read any previous version of the table.

- **Version 0** = before the MERGE (C-001 was in-progress)
- **Version 1** = after the MERGE (C-001 is resolved)

Both versions exist simultaneously. The old Parquet files were NOT deleted — they were just removed from the "active" list.

In [ ]:
# Read Version 0 (BEFORE the MERGE)
print("=== Version 0 (before MERGE) ===")
v0 = spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH)
v0.filter("call_id = 'C-001'").show()

In [ ]:
# Read Version 1 (AFTER the MERGE)
print("=== Version 1 (after MERGE) ===")
v1 = spark.read.format("delta").option("versionAsOf", 1).load(DELTA_PATH)
v1.filter("call_id = 'C-001'").show()

In [ ]:
# WHY: This is useful for debugging. "What did the data look like yesterday
# before the pipeline ran?" — just read the previous version.

# Compare: what changed between versions?
from pyspark.sql import functions as F

changed = (
    v0.alias("before")
    .join(v1.alias("after"), "call_id")
    .filter("before.status != after.status OR before.duration != after.duration")
    .select(
        "call_id",
        F.col("before.status").alias("old_status"),
        F.col("after.status").alias("new_status"),
        F.col("before.duration").alias("old_duration"),
        F.col("after.duration").alias("new_duration"),
    )
)

print("Records that changed:")
changed.show()

**You Should See:** One row — C-001 changed from `in-progress/0` to `resolved/480`.

### BigQuery Comparison

BigQuery has similar time travel, but limited to 7 days:

```sql
-- BigQuery time travel (read data from 1 hour ago)
SELECT * FROM silver.calls
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
WHERE call_id = 'C-001';
```

Delta Lake time travel is limited only by your VACUUM retention policy — you decide how long to keep old versions.

---

## 7. Schema Evolution

One day, the source system starts sending a new column: `agent_language`. Your table doesn't have this column yet.

- **Without** schema evolution: the write FAILS (Delta protects you from accidental schema changes)
- **With** `mergeSchema=true`: Delta adds the new column automatically

In [ ]:
# New data with an extra column: agent_language
new_data = spark.createDataFrame([
    Row(call_id="C-006", customer_id="CUST-105", status="resolved",
        duration=200, created_at=datetime(2026, 4, 14, 10, 0),
        agent_language="Spanish"),
])

print("New data schema (has agent_language):")
new_data.printSchema()

In [ ]:
# WHY: By default, Delta REJECTS writes with extra columns.
# This is GOOD — it catches accidental schema changes.
# But when you KNOW the new column is valid, enable mergeSchema.

new_data.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .save(DELTA_PATH)

print("Schema evolution: agent_language column added!")

In [ ]:
# Verify: table now has agent_language column
# Old records have null for agent_language (it didn't exist when they were written)
spark.read.format("delta").load(DELTA_PATH).show()

**You Should See:** 6 rows. C-006 has `agent_language=Spanish`. All other rows have `agent_language=null`.

This is **Version 2** of the table.

---

## 8. DESCRIBE HISTORY — Audit Trail

Every operation on a Delta table is logged. This is your audit trail — who did what, when.

In [ ]:
delta_table = DeltaTable.forPath(spark, DELTA_PATH)

# Show all operations
history = delta_table.history()
history.select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

**You Should See:** Three versions:

| Version | Operation | What Happened |
|---|---|---|
| 0 | WRITE | Initial load (5 records) |
| 1 | MERGE | Updated C-001 (in-progress → resolved) |
| 2 | WRITE | Appended C-006 with schema evolution |

---

## 9. VACUUM — Clean Up Old Files

Every MERGE and write creates NEW Parquet files. The old files stay on disk — that is what enables time travel. But old files cost storage.

VACUUM removes files that are no longer needed by any version within the retention window.

In [ ]:
# First, check how many files exist now
parquet_files = [f for f in os.listdir(DELTA_PATH) if f.endswith('.parquet')]
print(f"Parquet files before VACUUM: {len(parquet_files)}")
for f in sorted(parquet_files):
    print(f"  {f}")

In [ ]:
# WHY: In production, you would use retentionHours=168 (7 days).
# For this demo, we use 0 to show the effect immediately.
# retentionHours=0 means "delete everything not in the current version."

# Delta requires you to explicitly allow 0-hour retention
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

delta_table.vacuum(retentionHours=0)
print("VACUUM complete!")

In [ ]:
# Check files after VACUUM
parquet_files_after = [f for f in os.listdir(DELTA_PATH) if f.endswith('.parquet')]
print(f"Parquet files after VACUUM: {len(parquet_files_after)}")
for f in sorted(parquet_files_after):
    print(f"  {f}")

print(f"\nFiles removed: {len(parquet_files) - len(parquet_files_after)}")

**You Should See:** Fewer Parquet files. The old files from Version 0 (before the MERGE) are gone.

**WARNING:** After VACUUM, time travel to Version 0 will FAIL because the files are deleted:

```python
# This would fail now:
# spark.read.format("delta").option("versionAsOf", 0).load(DELTA_PATH)
# FileNotFoundException!
```

In production, use `retentionHours=168` (7 days) to keep a comfortable recovery window.

---

## 10. DELETE a Record

With plain Parquet, you cannot delete a single row — you must rewrite the entire file. With Delta Lake, DELETE works like a database.

In [ ]:
# WHY: DELETE is important for GDPR (General Data Protection Regulation).
# When a customer requests data deletion, you need to remove their records.
# With Delta, it's one line. With plain Parquet, it's a full rewrite.

delta_table = DeltaTable.forPath(spark, DELTA_PATH)

print(f"Rows before DELETE: {spark.read.format('delta').load(DELTA_PATH).count()}")

# Delete the missed call
delta_table.delete("call_id = 'C-003'")

print(f"Rows after DELETE: {spark.read.format('delta').load(DELTA_PATH).count()}")
spark.read.format("delta").load(DELTA_PATH).show()

**You Should See:** 5 rows (C-003 is gone). This is **Version 3**.

---

## Summary: What Delta Lake Gives You

| Operation | Plain Parquet | Delta Lake |
|---|---|---|
| Write safely (no corruption) | No — partial writes possible | Yes — ACID transactions |
| Update one row | Rewrite entire file | `MERGE` |
| Delete one row | Rewrite entire file | `DELETE` |
| See previous versions | Not possible | `option("versionAsOf", N)` |
| Add a column safely | Overwrite and hope | `mergeSchema=true` |
| Audit trail | Not available | `DESCRIBE HISTORY` |
| Clean up old data | Manual | `VACUUM` |

All of this with one change: `format("delta")` instead of `format("parquet")`.

---

## Next Steps

- [Lakehouse Formats - Concepts](../../playbooks/data/lakehouse-formats/02_Concepts.md) — Delta vs Iceberg vs Hudi comparison
- [Lakehouse Formats - How It Works](../../playbooks/data/lakehouse-formats/04_How_It_Works.md) — Transaction log internals
- [Lakehouse Formats - Decision Guide](../../playbooks/data/lakehouse-formats/10_Decision_Guide.md) — When to use Delta vs BigQuery native
- [ETL Patterns](../../playbooks/data/etl-patterns/README.md) — CDC, incremental loads, dead letter queues

---

[Community](https://www.skool.com/deliverymomentum) | [Book a 1:1](https://calendly.com/sunil-mogadati/connect)